# Lab 13 · Final Writeup · turn a semester of code into a paper

The final deliverable is a short paper describing what you built, what you measured, and what it means. Not a lab report; a paper. Two-column, six pages, with the writing quality you'd want a colleague to read. This notebook produces the LaTeX skeleton, embeds your figures, and lists the pieces you need to write.

**Prerequisites.** Lab 12 (finished implementation + measurements).

**Builds toward.** Nothing after — this is the finale.

> **📚 Where to look when you're stuck**
>
> - [**ACM template** (acmart)](https://www.overleaf.com/latex/templates/association-for-computing-machinery-acm-sig-proceedings-template/bmvfhcdnxfty) — one of the standard HPC paper templates
> - [**How to write a good conference paper (Simon Peyton Jones)**](https://simon.peytonjones.org/great-research-paper/)
> - [**Zotero**](https://www.zotero.org/) — free citation manager



## How this notebook works

Same three surfaces as prior labs: **[Hub]**, **[Hub -> cluster]**, **[cluster compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab13", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab13 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> cluster] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab13 dir ready')


## Part 1 · Structure · the six-section paper

| Section | Purpose | Length |
|---|---|---|
| **Abstract** | 5 sentences: problem, approach, what you found, what it means, contribution | ¼ page |
| **Introduction** | Why this problem, what's hard, what you did, roadmap | 1 page |
| **Background / Related work** | The physics + algorithm + related implementations | 1 page |
| **Implementation** | How you built it, choices, code snippets if illustrative | 1.5 pages |
| **Results** | Every figure with interpretation; strong + weak scaling | 2 pages |
| **Conclusion + future work** | What you learned; two ideas someone could pick up | ½ page |

Six pages, two-column, 10pt. Every figure must have a paragraph explaining what it shows *and why it matters*. Every claim ('runs 3x faster') needs a citation to a figure or table.


In [ ]:
# [Hub] Drop a skeleton .tex file for the student to fill in.
tex = '''\\documentclass[sigconf,10pt]{acmart}\n\\title{Your Title Here}\n\\author{Your Name}\n\\begin{document}\n\\begin{abstract}\nFive-sentence pitch.\n\\end{abstract}\n\\maketitle\n\\section{Introduction}\nOne page.\n\\section{Background}\nOne page.\n\\section{Implementation}\n1.5 pages.\n\\section{Results}\nTwo pages of figures and interpretation.\n\\begin{figure}\\includegraphics{figures/rooflineCrux.pdf}\\caption{...}\\end{figure}\n\\section{Conclusion}\nHalf page.\n\\end{document}\n'''
(labDir/'paper.tex').write_text(tex)
print('wrote paper.tex skeleton - open in your favorite editor and start writing')


In [ ]:
checkpoint("Part 1 - skeleton in place", [
    check("paper.tex exists", fileExists(str(labDir/'paper.tex'))),
])


## Part 2 · Gather every figure

Copy every PDF you produced this semester into `~/lab13/figures/`. Not every one goes in the paper — pick the 4-6 most informative. Every included figure must have PDF resolution (not PNG), a house-style-consistent look (`applyHouseStyle`), and a self-contained caption.


In [ ]:
# [Hub] Gather figure paths from your other labs.
import glob
figDir = labDir / 'figures'; figDir.mkdir(exist_ok=True)
sources = sorted(glob.glob(str(labDir.parent/'lab*/figures/*.pdf')))
for src in sources:
    dst = figDir / pathlib.Path(src).name
    if not dst.exists():
        import shutil; shutil.copy(src, dst)
print(f'gathered {len(list(figDir.glob("*.pdf")))} figures into {figDir}')


In [ ]:
checkpoint("Part 2 - figures gathered", [
    check("figures dir has PDFs",
          lambda: (len(list((labDir/'figures').glob('*.pdf'))) > 0,
                   f'{len(list((labDir/"figures").glob("*.pdf")))} PDFs')),
])


## Part 3 · Consolidate `timings.csv`

Bring every measurement from every lab into one master CSV. This is your **paper's raw data** — future readers should be able to reproduce every plot from this one file.


In [ ]:
# [Hub] Concatenate all lab timings.csv files.
import pandas as pd
srcs = sorted(glob.glob(str(labDir.parent/'lab*/out/timings.csv')))
dfs = [pd.read_csv(s) for s in srcs]
master = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
master.to_csv(labDir/'timingsMaster.csv', index=False)
print(f'master CSV: {len(master)} rows from {len(srcs)} labs')


In [ ]:
checkpoint("Part 3 - master CSV", [
    check("timingsMaster.csv exists", fileExists(str(labDir/'timingsMaster.csv'))),
])


## Part 4 · Draft each section

Two-hour writing session per section. Draft, don't polish. In order: Introduction → Results → Implementation → Background → Conclusion → Abstract (yes, abstract last — you can't summarize what you don't yet have written).

**The abstract is the hardest paragraph in the paper.** Write it after the rest is done, and give it 30-60 minutes on its own.


In [ ]:
checkpoint("Part 4 - draft complete", [
    check("paper.tex has content",
          fileNonEmpty(str(labDir/'paper.tex'), minLines=20)),
])


## Part 5 · Compile, revise, revise, revise

Compile with `pdflatex paper.tex` (or `latexmk paper.tex`). Read the PDF as a stranger would. Cut anything that isn't earning its space.

**The single most important edit**: for every figure, ask *would someone who hadn't done this course understand what this shows?* If not, the caption needs to grow.


In [ ]:
# [Hub] Optional: compile locally if TeX Live is installed.
try:
    out, rc = runShell(f'cd {labDir} && pdflatex -interaction=nonstopmode paper.tex 2>&1 | tail -5')
    print(out)
except Exception as e:
    print(f'no local pdflatex - use Overleaf or install TeX Live: {e}')


In [ ]:
checkpoint("Part 5 - paper compiles", [
    check("paper.pdf exists (compiled)", fileExists(str(labDir/'paper.pdf'))),
])


## Part 6 · Ship

Submit the PDF + the `timingsMaster.csv` + a zipfile of your source tree. **Congratulations.** You wrote and measured a real HPC code across CPU, OpenMP, MPI, GPU, and multi-GPU, and produced a paper about it. That's what a graduate-level HPC course is *for*.


## Wrap up

Moved the spine forward one lab.


### Lab scorecard


In [ ]:
labSummary("Final Writeup")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("Final Writeup")
